In [0]:
pip install Faker

In [0]:
# =============================================================
# NOTEBOOK:  01_bronze_batch_ingestion_customers
# PURPOSE:   Auto Loader pipeline for CRM customer data
# SOURCE:    CRM System → CSV files
# TARGET:    Bronze Delta Table (bronze/crm/customers/)
# AUTHOR:    Your Name
# DATE:      Today's date
# PROJECT:   RetailMax Lakehouse
# =============================================================

import json
from pyspark.sql import functions as F
from pyspark.sql.types import *
from datetime import datetime
from faker import Faker
import random
import uuid



# ── Load Config ───────────────────────────────────────────────
config_path = "/Workspace/Repos/retailmax-lakehouse/retailmax-lakehouse/configs/dev/config.json"

with open(config_path, 'r') as f:
    config = json.load(f)

# ── Storage Paths ─────────────────────────────────────────────
storage_account = config['storage']['account_name']
bronze_path     = config['storage']['bronze_path']
scope_name      = config['secret_scope']

# ── Retrieve Secrets ──────────────────────────────────────────
client_id     = dbutils.secrets.get(scope=scope_name, key="sp-client-id")
client_secret = dbutils.secrets.get(scope=scope_name, key="sp-client-secret")
tenant_id     = dbutils.secrets.get(scope=scope_name, key="sp-tenant-id")

# ── Configure Spark for ADLS ──────────────────────────────────
spark.conf.set(
    f"fs.azure.account.auth.type.{storage_account}.dfs.core.windows.net",
    "OAuth"
)
spark.conf.set(
    f"fs.azure.account.oauth.provider.type.{storage_account}.dfs.core.windows.net",
    "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider"
)
spark.conf.set(
    f"fs.azure.account.oauth2.client.id.{storage_account}.dfs.core.windows.net",
    client_id
)
spark.conf.set(
    f"fs.azure.account.oauth2.client.secret.{storage_account}.dfs.core.windows.net",
    client_secret
)
spark.conf.set(
    f"fs.azure.account.oauth2.client.endpoint.{storage_account}.dfs.core.windows.net",
    f"https://login.microsoftonline.com/{tenant_id}/oauth2/token"
)

print("=" * 60)
print("RetailMax - Bronze Batch Ingestion: Customers")
print("=" * 60)
print(f"✅ Config loaded")
print(f"✅ Secrets retrieved")
print(f"✅ ADLS configured")
print(f"   Bronze Path: {bronze_path}")
print("=" * 60)



In [0]:
source_path = bronze_path + 'raw_ingest/crm/customers/'
target_path = bronze_path + "tables/crm/customers/"
checkpoint_path = bronze_path + "_checkpoints/crm/customers/"
schema_path = bronze_path + "_schema/crm/customers/"
bad_records_path = bronze_path + "_quarantine/crm/customers/"


In [0]:
fake = Faker()

# Pattern for generating N records
def generate_customers(start_id,n=1000):
    customers = []
    
    for i in range(start_id, start_id+n):
        customers.append({
            "customer_id": i,
            "first_name":  fake.first_name(),
            "last_name":  fake.last_name(),
            "email": fake.email() if random.random() > 0.03 else None,
            "phone": fake.phone_number() if random.random() > 0.1 else random.choice(["-9894838444", "999", None]),
            "country": fake.country() if random.random() > 0.02 else None,
            "city": fake.city() if random.random() > 0.02 else None,
            "segment": random.choice(["Premium", "Standard", "Basic", "VIP"]),
            "loyalty_points": random.randint(0, 50000),
            "created_date": fake.date_between(
                start_date="-2y",
                end_date="today"
            ).isoformat(),
            "is_active": random.choice([1,1,1,1,0]),
            "age": random.randint(18,80) if random.random() > 0.03 else random.choice([-1, None])
        })
    
    return customers

# Generate and convert to DataFrame
data1 = generate_customers(start_id=1,n=1000)
data2 = generate_customers(start_id=1001,n=1000)
df1 = spark.createDataFrame(data1)
df2 = spark.createDataFrame(data2)
df1.show(5)
df2.show(5)

# Save DataFrame as CSV
(df1
    .write
    .mode("overwrite")
    .option("header", "true")
    .csv(source_path + "customers_2026_07_06_10/")
)

(df2
    .write
    .mode("overwrite")
    .option("header", "true")
    .csv(source_path + "customers_2026_07_06_11/")
)

In [0]:
print("Source folders:")
display(dbutils.fs.ls(source_path))

print("Files inside first batch:")
display(dbutils.fs.ls(source_path + "customers_2026_07_06_10/"))

In [0]:
# Reading with Auto Loader
df = spark.readStream\
    .format("cloudFiles")\
    .option("cloudFiles.format", "csv")\
    .option("cloudFiles.schemaLocation", schema_path)\
    .option("header", "true")\
    .option("inferSchema", "true")\
    .load(source_path)

# Metadata columns
df = (df
    .withColumn("_source_file", F.input_file_name())
    .withColumn("_load_timestamp", F.current_timestamp())
    .withColumn("_load_date", F.current_date())
    .withColumn("_batch_id", F.lit(uuid.uuid4().hex))
)

# Writing stream
df.writeStream\
    .format("delta")\
    .option("checkpointLocation", checkpoint_path)\
    .trigger(availableNow=True)\
    .partitionBy("_load_date")\
    .start(target_path)\
    .awaitTermination()

In [0]:
df = spark.read.format("delta").load(target_path)
print(df.count())
df.show(5)
df.printSchema()